In [88]:
# 基础工具
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [89]:
# 读取损失数量原始表
df = pd.read_excel('../data/raw/loss_qty_raw.xlsx')
df.head()

,defect_material_no,loss_qty,owner,raw_process_name,month
0,1373223-003,7,石狮九牧,饮用不锈钢机加,2025-08
1,1385460-004,111,石狮九牧,饮用,2025-11
2,14045305-002,144,石狮九牧,饮用不锈钢抛光,2025-01
3,1393008-004,48,石狮九牧,饮用,2025-11
4,14037920-003,36,石狮九牧,饮用,2025-03


In [90]:
df.shape

(14918, 5)

In [91]:
# 原始表体检：缺失值、筛选字段分布、损失数量分布
df.isna().sum()
df['owner'].value_counts()
df['raw_process_name'].value_counts()
df[df['loss_qty'] <= 0]
df['loss_qty'].describe()
df.sort_values('loss_qty', ascending=False).head(10)

,defect_material_no,loss_qty,owner,raw_process_name,month
1761,14089175-004,1519,石狮九牧,饮用,2025-05
8735,1391283-002,1243,石狮九牧,饮用,2025-07
955,1391283-004,1126,石狮九牧,饮用,2025-09
9839,14089175-004,1087,石狮九牧,饮用,2025-06
1674,1391283-002,1066,石狮九牧,饮用不锈钢抛光,2025-06
9169,1391283-004,996,石狮九牧,饮用,2025-05
7705,1391283-002,990,石狮九牧,饮用,2025-04
4088,14089175-003,962,石狮九牧,饮用不锈钢机加,2025-05
13299,1325111-002,950,石狮九牧,饮用不锈钢抛光,2025-06
6622,14089175-004,940,石狮九牧,饮用,2025-10


In [92]:
df['raw_process_name'].value_counts()

raw_process_name
饮用         9008
饮用不锈钢机加    2337
饮用不锈钢抛光    2146
仓储          303
开发          297
总装          288
锌合金         282
ABS         257
Name: count, dtype: int64

# 损失数量表清洗规则

1. 输入表：
loss_qty_raw.xlsx

2. 原始表一行代表什么：
一条产品在某个月、某责任归属、某原始工序类别下的损失数量记录。

3. 正式分析要保留的记录：
owner = 石狮九牧
raw_process_name 包含 饮用

4. 需要新增的字段：
sap_code：由 defect_material_no 去掉后缀得到
suffix：由 defect_material_no 的后缀得到
process：根据 suffix 映射得到标准工序

5. process 映射规则：
004 -> casting
003 -> machining
002 -> polishing

6. 清洗后表的一行代表什么：
一条已经过滤责任归属和饮用类别，并补充了 sap_code、suffix、process 的损失明细记录。

7. 后续还需要再聚合成什么粒度：
month + sap_code + process

In [93]:
# 只保留正式分析范围：本厂责任 + 饮用相关
loss_clean = df[(df['owner'] == '石狮九牧') & (df['raw_process_name'].str.contains('饮用', na=False))].copy()

In [94]:
# 从 defect_material_no 拆出产品编码和工序后缀
loss_clean[['sap_code', 'suffix']] = loss_clean['defect_material_no'].str.split('-', expand=True)

In [95]:
loss_clean.head()

,defect_material_no,loss_qty,owner,raw_process_name,month,sap_code,suffix
0,1373223-003,7,石狮九牧,饮用不锈钢机加,2025-08,1373223,003
1,1385460-004,111,石狮九牧,饮用,2025-11,1385460,004
2,14045305-002,144,石狮九牧,饮用不锈钢抛光,2025-01,14045305,002
3,1393008-004,48,石狮九牧,饮用,2025-11,1393008,004
4,14037920-003,36,石狮九牧,饮用,2025-03,14037920,003


In [96]:
# 根据后缀生成标准工序，不能直接使用 raw_process_name
process_map = {
    '004': 'casting',
    '003': 'machining',
    '002': 'polishing'
}
loss_clean['process'] = loss_clean['suffix'].map(process_map)

In [97]:
loss_clean['process'].value_counts(dropna=False) #检查是否有process为NaN的行，即不属于铸造、机加、抛光这几个工序的编码

process
machining    4241
casting      4137
polishing    3940
Name: count, dtype: int64

In [98]:
loss_clean.head()

,defect_material_no,loss_qty,owner,raw_process_name,month,sap_code,suffix,process
0,1373223-003,7,石狮九牧,饮用不锈钢机加,2025-08,1373223,003,machining
1,1385460-004,111,石狮九牧,饮用,2025-11,1385460,004,casting
2,14045305-002,144,石狮九牧,饮用不锈钢抛光,2025-01,14045305,002,polishing
3,1393008-004,48,石狮九牧,饮用,2025-11,1393008,004,casting
4,14037920-003,36,石狮九牧,饮用,2025-03,14037920,003,machining


In [99]:
# 聚合到后续事实表需要的粒度：month + sap_code + process
loss_by_process = (
    loss_clean
    .groupby(['month', 'sap_code', 'process'], as_index=False)
    .agg(loss_qty=('loss_qty', 'sum'))
)
loss_by_process.head()

,month,sap_code,process,loss_qty
0,2025-01,1300175,machining,26
1,2025-01,1300391,casting,5
2,2025-01,1300493,casting,22
3,2025-01,1300493,machining,21
4,2025-01,1300493,polishing,28


In [100]:
loss_by_process.shape

(12318, 4)

In [101]:
loss_by_process['loss_qty'].sum()

np.int64(698491)

In [102]:
# 读取生产数量原始表；此时仍然是宽表
production_raw = pd.read_excel('../data/raw/production_qty_raw.xlsx')
production_raw.shape

(6000, 7)

In [103]:
production_raw.head()

,month,sap_code,product_name,product_category,casting_production_qty,machining_production_qty,polishing_production_qty
0,2025-01,1300175,厨房龙头-M0001,厨房龙头,3520,3380,2988
1,2025-02,1300175,厨房龙头-M0001,厨房龙头,2733,2538,2587
2,2025-03,1300175,厨房龙头-M0001,厨房龙头,5068,4235,3972
3,2025-04,1300175,厨房龙头-M0001,厨房龙头,3590,3155,3743
4,2025-05,1300175,厨房龙头-M0001,厨房龙头,4562,4302,4490


## 2. 生产数量表宽转长

目标：把 `production_qty_raw.xlsx` 从宽表转成长表，得到 `production_long`。

业务原因：`loss_by_process` 已经是 `month + sap_code + process` 粒度，生产表也必须变成同样粒度，后面才能 merge。

结果字段：

```text
month, sap_code, product_name, product_category, process, production_qty
```


In [104]:
# 将生产数量宽表转成长表，使粒度与 loss_by_process 一致
production_long = pd.melt(
    production_raw,
    id_vars=['month', 'sap_code', 'product_name', 'product_category'],
    value_vars=['casting_production_qty', 'machining_production_qty', 'polishing_production_qty'],
    var_name='process',
    value_name='production_qty'
)

production_long.head()

,month,sap_code,product_name,product_category,process,production_qty
0,2025-01,1300175,厨房龙头-M0001,厨房龙头,casting_production_qty,3520
1,2025-02,1300175,厨房龙头-M0001,厨房龙头,casting_production_qty,2733
2,2025-03,1300175,厨房龙头-M0001,厨房龙头,casting_production_qty,5068
3,2025-04,1300175,厨房龙头-M0001,厨房龙头,casting_production_qty,3590
4,2025-05,1300175,厨房龙头-M0001,厨房龙头,casting_production_qty,4562


In [105]:
# 把 melt 后的原始列名清洗成标准工序名，方便后续按 process 合并
production_long['process'] = production_long['process'].replace({
    'casting_production_qty': 'casting',
    'machining_production_qty': 'machining',
    'polishing_production_qty': 'polishing'
})

production_long.head()

,month,sap_code,product_name,product_category,process,production_qty
0,2025-01,1300175,厨房龙头-M0001,厨房龙头,casting,3520
1,2025-02,1300175,厨房龙头-M0001,厨房龙头,casting,2733
2,2025-03,1300175,厨房龙头-M0001,厨房龙头,casting,5068
3,2025-04,1300175,厨房龙头-M0001,厨房龙头,casting,3590
4,2025-05,1300175,厨房龙头-M0001,厨房龙头,casting,4562


In [106]:
# 生产数量长表检查
production_check = pd.Series({
    'production_raw_rows': production_raw.shape[0],
    'production_long_rows': production_long.shape[0],
    'process_values': ', '.join(sorted(production_long['process'].unique())),
    'missing_production_qty': production_long['production_qty'].isna().sum(),
    'zero_production_qty': production_long['production_qty'].eq(0).sum(),
    'raw_total_production_qty': production_raw[
        ['casting_production_qty', 'machining_production_qty', 'polishing_production_qty']
    ].sum().sum(),
    'long_total_production_qty': production_long['production_qty'].sum()
})

production_check

production_raw_rows                                   6000
production_long_rows                                 18000
process_values               casting, machining, polishing
missing_production_qty                                   0
zero_production_qty                                   1320
raw_total_production_qty                          99922595
long_total_production_qty                         99922595
dtype: object

## 3. 工序单价表宽转长

目标：把 `process_price_raw.xlsx` 转成长表，得到 `price_long`。

结果字段：

```text
sap_code, process, unit_price
```

业务原因：后面要按 `sap_code + process` 把单价合并到事实表，用来计算 `loss_amount = loss_qty * unit_price`。


In [107]:
# 读取工序单价原始表；此时仍然是宽表
price_raw = pd.read_excel('../data/raw/process_price_raw.xlsx')
price_raw.head()

,sap_code,casting_unit_price,machining_unit_price,polishing_unit_price
0,1300175,12.21,21.58,49.01
1,1300391,12.44,28.39,11.82
2,1300493,16.62,13.09,50.77
3,1301111,14.20,0.00,28.86
4,1301195,21.66,14.63,7.56


In [108]:
# 将工序单价宽表转成长表，方便后续按 sap_code + process 合并
price_long = price_raw.melt(
    id_vars=['sap_code'],
    var_name='process',
    value_name='unit_price'
)

price_long.head()

,sap_code,process,unit_price
0,1300175,casting_unit_price,12.21
1,1300391,casting_unit_price,12.44
2,1300493,casting_unit_price,16.62
3,1301111,casting_unit_price,14.20
4,1301195,casting_unit_price,21.66


In [109]:
# 把 melt 后的原始列名清洗成标准工序名
price_long['process'] = price_long['process'].replace({
    'casting_unit_price': 'casting',
    'machining_unit_price': 'machining',
    'polishing_unit_price': 'polishing'
})

price_long.head()

,sap_code,process,unit_price
0,1300175,casting,12.21
1,1300391,casting,12.44
2,1300493,casting,16.62
3,1301111,casting,14.20
4,1301195,casting,21.66


In [110]:
# 工序单价长表检查
price_check = pd.Series({
    'price_raw_rows': price_raw.shape[0],
    'price_long_rows': price_long.shape[0],
    'process_values': ', '.join(sorted(price_long['process'].unique())),
    'missing_unit_price': price_long['unit_price'].isna().sum(),
    'zero_unit_price': price_long['unit_price'].eq(0).sum(),
    'duplicated_sap_code_process': price_long.duplicated(['sap_code', 'process']).sum()
})

price_check

price_raw_rows                                           500
price_long_rows                                         1500
process_values                 casting, machining, polishing
missing_unit_price                                         0
zero_unit_price                                          118
duplicated_sap_code_process                                0
dtype: object

## 4. 合并生成工序事实表

目标：合并三张长表，生成 `defect_loss_fact`。

输入表：

```text
loss_by_process：month + sap_code + process + loss_qty
production_long：month + sap_code + product_name + product_category + process + production_qty
price_long：sap_code + process + unit_price
```

建议从 `production_long` 出发合并，因为它包含每月每个产品每道工序的完整生产框架；没有损失的工序，`loss_qty` 后面补 0。


In [111]:
# 合并前统一 sap_code 类型，避免字符串编码和数字编码匹配不上
loss_by_process['sap_code'] = loss_by_process['sap_code'].astype(str)
production_long['sap_code'] = production_long['sap_code'].astype(str)
price_long['sap_code'] = price_long['sap_code'].astype(str)

In [112]:
# 从 production_long 出发合并损失数量；无损失的工序 loss_qty 补 0
production_loss = production_long.merge(
    loss_by_process,
    on=['month', 'sap_code', 'process'],
    how='left'
)

production_loss['loss_qty'] = production_loss['loss_qty'].fillna(0).astype(int)

production_loss.head()

,month,sap_code,product_name,product_category,process,production_qty,loss_qty
0,2025-01,1300175,厨房龙头-M0001,厨房龙头,casting,3520,0
1,2025-02,1300175,厨房龙头-M0001,厨房龙头,casting,2733,33
2,2025-03,1300175,厨房龙头-M0001,厨房龙头,casting,5068,45
3,2025-04,1300175,厨房龙头-M0001,厨房龙头,casting,3590,0
4,2025-05,1300175,厨房龙头-M0001,厨房龙头,casting,4562,41


In [113]:
# 合并工序单价，并计算工序损失金额
defect_loss_fact = production_loss.merge(
    price_long,
    on=['sap_code', 'process'],
    how='left'
)

defect_loss_fact['loss_amount'] = defect_loss_fact['loss_qty'] * defect_loss_fact['unit_price']

defect_loss_fact.head()

,month,sap_code,product_name,product_category,process,production_qty,loss_qty,unit_price,loss_amount
0,2025-01,1300175,厨房龙头-M0001,厨房龙头,casting,3520,0,12.21,0.00
1,2025-02,1300175,厨房龙头-M0001,厨房龙头,casting,2733,33,12.21,402.93
2,2025-03,1300175,厨房龙头-M0001,厨房龙头,casting,5068,45,12.21,549.45
3,2025-04,1300175,厨房龙头-M0001,厨房龙头,casting,3590,0,12.21,0.00
4,2025-05,1300175,厨房龙头-M0001,厨房龙头,casting,4562,41,12.21,500.61


### 检查项

```text
defect_loss_fact 行数应等于 production_long 行数：18000
loss_qty 缺失应为 0
unit_price 缺失应为 0
loss_qty 总和应等于 loss_by_process 的 loss_qty 总和
loss_amount 不应为空
month + sap_code + process 不应重复
```


In [114]:
# 工序事实表检查
fact_check = pd.Series({
    'fact_rows': defect_loss_fact.shape[0],
    'production_long_rows': production_long.shape[0],
    'missing_loss_qty': defect_loss_fact['loss_qty'].isna().sum(),
    'missing_unit_price': defect_loss_fact['unit_price'].isna().sum(),
    'fact_total_loss_qty': defect_loss_fact['loss_qty'].sum(),
    'source_total_loss_qty': loss_by_process['loss_qty'].sum(),
    'missing_loss_amount': defect_loss_fact['loss_amount'].isna().sum(),
    'duplicated_month_sap_process': defect_loss_fact.duplicated(['month', 'sap_code', 'process']).sum()
})

fact_check

fact_rows                        18000
production_long_rows             18000
missing_loss_qty                     0
missing_unit_price                   0
fact_total_loss_qty             698491
source_total_loss_qty           698491
missing_loss_amount                  0
duplicated_month_sap_process         0
dtype: int64

### 批改标注：工序事实表合并

方向正确：从 `production_long` 出发，先合并损失数量，再合并单价，这个顺序是对的。这样能保留没有损失的产品工序记录，后面不良率才不会只剩“有损失”的部分。

已修正：

```text
1. 三张表的 sap_code 都显式转成字符串。
2. 中间表变量从 a 改成 production_loss，含义更清楚。
3. loss_qty 补 0 后转成整数。
4. 重复检查改成 month + sap_code + process。
```

注意：`sap_code + process` 跨月份重复是正常的，因为同一个产品同一道工序会出现在 12 个月。事实表真正的一行粒度是 `month + sap_code + process`。


## 5. 工序指标与数据质量初查

目标：在 `defect_loss_fact` 上新增工序不良率，并做第一批数据质量检查。

暂时不要算产品综合不良率，那是产品月汇总阶段再做。


In [115]:
# 标记该产品该工序是否有本厂生产数据
defect_loss_fact['has_process_data'] = defect_loss_fact['production_qty'] > 0

In [116]:
# 计算工序不良率；无本厂生产数据时先留空，避免除以 0
defect_loss_fact['defect_rate'] = np.where(
    defect_loss_fact['has_process_data'],
    defect_loss_fact['loss_qty'] / defect_loss_fact['production_qty'],
    np.nan
)

In [117]:
# 第一批数据质量检查
dq_check = pd.Series({
    'defect_rate_inf_rows': np.isinf(defect_loss_fact['defect_rate']).sum(),
    'loss_qty_gt_production_qty_rows': (defect_loss_fact['loss_qty'] > defect_loss_fact['production_qty']).sum(),
    'loss_qty_gt0_unit_price_eq0_rows': (
        (defect_loss_fact['loss_qty'] > 0) & (defect_loss_fact['unit_price'] == 0)
    ).sum(),
    'loss_qty_gt0_production_qty_eq0_rows': (
        (defect_loss_fact['loss_qty'] > 0) & (defect_loss_fact['production_qty'] == 0)
    ).sum()
})

dq_check
#当前初查结果：没有损失数量大于生产数量；没有有损失但生产数量为 0；有 78 行存在有损失但单价为 0，需要作为数据质量问题保留。

defect_rate_inf_rows                     0
loss_qty_gt_production_qty_rows          0
loss_qty_gt0_unit_price_eq0_rows        78
loss_qty_gt0_production_qty_eq0_rows     0
dtype: int64

In [118]:
# 查看有损失但单价为 0 的记录，后续进入数据质量问题表
zero_price_loss_rows = defect_loss_fact[
    (defect_loss_fact['loss_qty'] > 0) & (defect_loss_fact['unit_price'] == 0)
]

zero_price_loss_rows.head()

,month,sap_code,product_name,product_category,process,production_qty,loss_qty,unit_price,loss_amount,has_process_data,defect_rate
252,2025-01,1306651,浴室挂件-M0022,浴室挂件,casting,2185,20,0.0,0.0,True,0.009153
253,2025-02,1306651,浴室挂件-M0022,浴室挂件,casting,2007,16,0.0,0.0,True,0.007972
254,2025-03,1306651,浴室挂件-M0022,浴室挂件,casting,3273,26,0.0,0.0,True,0.007944
255,2025-04,1306651,浴室挂件-M0022,浴室挂件,casting,3608,34,0.0,0.0,True,0.009424
256,2025-05,1306651,浴室挂件-M0022,浴室挂件,casting,2620,13,0.0,0.0,True,0.004962


## 6. 导出事实表 CSV

目标：把 `defect_loss_fact` 导出成 CSV，后面用 MySQL Workbench 导入。


In [120]:
# 导出标准事实表：保留中文展示字段和缺失工序的空 defect_rate
defect_loss_fact.to_csv(
    "../data/processed/defect_loss_fact.csv",
    index=False,
    encoding="utf-8"
)

# Workbench 导入兼容版：保持 has_process_data 标记，空 defect_rate 写为 0
mysql_category_map = {
    "厨房龙头": "kitchen_faucet",
    "淋浴龙头": "shower_faucet",
    "五金配件": "hardware_accessory",
    "浴室挂件": "bathroom_accessory"
}

defect_loss_fact_ascii = defect_loss_fact.copy()
defect_loss_fact_ascii["product_category"] = defect_loss_fact_ascii["product_category"].map(mysql_category_map)
defect_loss_fact_ascii["product_name"] = (
    defect_loss_fact_ascii["product_category"]
    + "-"
    + defect_loss_fact_ascii["sap_code"].str.extract(r"(M\d+)$")[0]
)
defect_loss_fact_ascii["defect_rate"] = defect_loss_fact_ascii["defect_rate"].fillna(0)
defect_loss_fact_ascii.to_csv(
    "../data/processed/defect_loss_fact_ascii.csv",
    index=False,
    encoding="ascii"
)

下一步：在 MySQL Workbench 导入 `data/processed/defect_loss_fact_ascii.csv`，表名使用 `defect_loss_fact`。导入版中无本厂工序数据仍由 `has_process_data = False` 标记，`defect_rate` 为兼容导入而写为 `0`。